# Lab 06: DCGAN on MNIST

**Module 06** | Read `notes/06-dcgan-gan-training.pdf` before running this notebook.

Apply DCGAN architectural rules (strided convolutions, batch norm, no pooling) and compare sample quality to the vanilla GAN lab.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## DCGAN on MNIST

Deep Convolutional GANs (DCGANs) replace fully connected layers with strided convolutions in the discriminator and transposed convolutions in the generator. Radford et al. distilled several architectural rules that stabilize GAN training: use batch normalization in both networks (except the generator output and discriminator input), employ LeakyReLU activations, avoid pooling layers in favor of strided convolutions, and remove fully connected hidden layers so the network is fully convolutional.

Compared with the shallow MLP GAN in Lab 05, DCGANs exploit spatial structure in 28×28 digits and typically produce sharper samples with fewer parameters in the dense layers.


MNIST images are scaled to `[−1, 1]` so they match the generator's `Tanh` output range. We train for a handful of epochs, enough to see digit-like structure without long GPU time, and save a grid of generated samples at the end.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "datasets" / "mnist"
BATCH_SIZE = 128
LATENT_DIM = 100
EPOCHS = 6
LR = 2e-4

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"MNIST train images: {len(train_set):,} | Batches per epoch: {len(loader)}")


The generator upsamples a 100-dimensional noise vector through four transposed convolutions with batch normalization and ReLU. The final layer uses `Tanh` and produces a 32×32 map; we center-crop to 28×28 to match MNIST resolution.

The discriminator mirrors this with strided convolutions (no max-pooling), LeakyReLU(0.2), and batch norm on all but the first layer. The output is a single logit per image, we pair it with `BCEWithLogitsLoss` instead of a sigmoid for numerical stability.


In [ ]:
class DCGANGenerator(nn.Module):
    def __init__(self, latent_dim: int = 100, ngf: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, 1, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.net(z.view(z.size(0), -1, 1, 1))
        return x[:, :, 2:30, 2:30]


class DCGANDiscriminator(nn.Module):
    def __init__(self, ndf: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(-1)


G = DCGANGenerator(LATENT_DIM).to(device)
D = DCGANDiscriminator().to(device)
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
print(f"Generator params: {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")


Training alternates discriminator and generator updates with label smoothing avoided, we use hard 0/1 targets. The generator is trained to make the discriminator classify fakes as real (`label=1`), which is the standard non-saturating generator objective.

Adam with `lr=2e-4` and `betas=(0.5, 0.999)` matches the DCGAN paper and Lab 05 for fair comparison.


In [ ]:
g_losses, d_losses = [], []
fixed_noise = torch.randn(64, LATENT_DIM, device=device)

for epoch in range(1, EPOCHS + 1):
    g_epoch, d_epoch = 0.0, 0.0
    for real, _ in loader:
        real = real.to(device)
        batch = real.size(0)
        real_labels = torch.ones(batch, device=device)
        fake_labels = torch.zeros(batch, device=device)

        D.zero_grad()
        d_real = D(real)
        loss_d_real = criterion(d_real, real_labels)
        z = torch.randn(batch, LATENT_DIM, device=device)
        fake = G(z).detach()
        d_fake = D(fake)
        loss_d_fake = criterion(d_fake, fake_labels)
        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        opt_d.step()

        G.zero_grad()
        z = torch.randn(batch, LATENT_DIM, device=device)
        fake = G(z)
        d_out = D(fake)
        loss_g = criterion(d_out, real_labels)
        loss_g.backward()
        opt_g.step()

        g_epoch += loss_g.item()
        d_epoch += loss_d.item()

    g_losses.append(g_epoch / len(loader))
    d_losses.append(d_epoch / len(loader))
    print(f"Epoch {epoch:02d}/{EPOCHS}, G loss {g_losses[-1]:.4f} | D loss {d_losses[-1]:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), g_losses, label="Generator", marker="o")
plt.plot(range(1, EPOCHS + 1), d_losses, label="Discriminator", marker="s")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("DCGAN training losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


A grid of samples from fixed noise vectors reveals mode coverage and sharpness. DCGAN outputs should look more structured than the MLP GAN from Lab 05 after similar epoch counts, compare digit edges and class diversity when you revisit both notebooks.


In [ ]:
G.eval()
with torch.no_grad():
    fake_batch = G(fixed_noise).cpu()

grid = make_grid(fake_batch, nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("DCGAN samples (fixed noise)")
plt.axis("off")
plt.tight_layout()
plt.show()
